## Loading Data

In [9]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

access the drive that has the data

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from zipfile import ZipFile

In [12]:
images_zip_path = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images.zip'

# images_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images'

In [ ]:
# !unzip {images_zip_path} -d /content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/

In [ ]:
# labels_zip_path = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/labels.zip'



In [ ]:
# !unzip {labels_zip_path} -d /content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/labels/

In [ ]:
# example_image = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/MEN-Denim-id_00000080-01_7_additional_segm.png'

In [ ]:
# example_image

In [ ]:
# from PIL import Image
# import numpy as np

# segm = Image.open(example_image)
# segm = np.array(segm) # shape: [750, 1101]


## EDA: Labels

In [13]:
shape_annon = ['sleeve length', 'lower clothing length', 'socks', 'hat', 'glasses', 'neckwear', 'wrist wearing', 'ring', 'waist accessories', 'neckline', 'outer clothing a cardigan?', 'upper clothing covering navel']


In [14]:
shap_annons = {
    "sleeve_length": {0: "sleeveless", 1: "short-sleeve", 2: "medium-sleeve", 3: "long-sleeve", 4: "not long-sleeve", 5: "NA"},
    "lower_clothing_length": {0: "three-point", 1: "medium short", 2 :"three-quarter", 3: "long", 4: "NA"},
    "socks": {0: "no", 1: "socks", 2: "leggings", 3: "NA"},
    "hat": {0: "no", 1: "yes", 2: "NA"},
    "glasses": {0: "no", 1: "eyeglasses", 2: "sunglasses", 3: "have a glasses in hand or clothes", 4: "NA"},
    "neckwear": {0: "no", 1: "yes", 2: "NA"},
    "wrist_wearing": {0: "no", 1: "yes", 2: "NA"},
    "ring": {0: "no", 1: "yes", 2: "NA"},
    "waist_accessories": {0: "no", 1: "belt", 2: "have a clothing", 3: "hidden", 4: "NA"},
    "neckline": {0: "V-shape", 1: "square", 2: "round", 3: "standing", 4: "lapel", 5: "suspenders", 6: "NA"},
    "outer_clothing_cardigan": {0: "yes", 1: "no", 2: "NA"},
    "upper_clothing_covering_navel": {0: "no", 1: "yes", 2: "NA"}
}

top_modest = {
              'sleeve_length': {3:'long_sleeve'},
              'neckline': {2:'round', 3:'standing', 4:'lapel'},
              'outer_clothing_cardigan': {0:'yes'},
              'upper_clothing_covering_navel': {1:'yes'},
              }

# column number in shape_labels
top_modest = [3, [2,3,4], 0, 1]

bottom_modest = {
    'lower_clothing_length': {3:'long'},
    'socks': {1:'socks'}
    }

bottom_modest = [3, [1, 2]]


In [15]:
shape_labels = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/labels/labels/shape/shape_anno_all.txt'

In [16]:
shape_labels = pd.read_csv(shape_labels, sep=' ', header=None)


In [17]:
shape_labels

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,MEN-Denim-id_00000080-01_7_additional.jpg,5,3,0,0,0,0,0,0,3,2,1,1
1,MEN-Denim-id_00000089-01_7_additional.jpg,0,3,0,0,0,0,0,0,3,2,1,1
2,MEN-Denim-id_00000089-02_7_additional.jpg,3,3,0,0,0,0,0,0,3,4,1,1
3,MEN-Denim-id_00000089-03_7_additional.jpg,1,3,0,0,0,0,0,0,3,2,1,1
4,MEN-Denim-id_00000089-04_7_additional.jpg,3,3,0,0,0,0,0,0,3,4,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42539,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,0,0,0,0,0,0,0,1,0,2,1,1
42540,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,0,0,3,0,0,0,0,1,0,2,1,1
42541,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,0,4,3,0,0,0,1,1,3,2,1,1
42542,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,0,4,3,0,0,0,1,1,3,6,2,2


In [18]:
shape_labels[['Men/Women', 'Clothing', 'id','img_num']] = shape_labels[0].str.split('-', expand=True)

In [19]:
shape_labels[['image_num', 'img_unit', 'img_side']] = shape_labels['img_num'].str.split('_', expand=True)

In [20]:
shape_labels[['clothing', 'sub_clothing']] = shape_labels['Clothing'].str.split('_', expand=True)

In [21]:
shape_annon = ['sleeve length', 'lower clothing length', 'socks', 'hat', 'glasses', 'neckwear', 'wrist wearing', 'ring', 'waist accessories', 'neckline', 'outer clothing a cardigan?', 'upper clothing covering navel']
# 1, 10, 12
top_modest = [3, [2,3,4], 1]

#2, 3
bottom_modest = [3, [1, 2]]


In [22]:
shape_labels

,0,1,2,3,4,5,6,7,8,9,...,12,Men/Women,Clothing,id,img_num,image_num,img_unit,img_side,clothing,sub_clothing
0,MEN-Denim-id_00000080-01_7_additional.jpg,5,3,0,0,0,0,0,0,3,...,1,MEN,Denim,id_00000080,01_7_additional.jpg,01,7,additional.jpg,Denim,None
1,MEN-Denim-id_00000089-01_7_additional.jpg,0,3,0,0,0,0,0,0,3,...,1,MEN,Denim,id_00000089,01_7_additional.jpg,01,7,additional.jpg,Denim,None
2,MEN-Denim-id_00000089-02_7_additional.jpg,3,3,0,0,0,0,0,0,3,...,1,MEN,Denim,id_00000089,02_7_additional.jpg,02,7,additional.jpg,Denim,None
3,MEN-Denim-id_00000089-03_7_additional.jpg,1,3,0,0,0,0,0,0,3,...,1,MEN,Denim,id_00000089,03_7_additional.jpg,03,7,additional.jpg,Denim,None
4,MEN-Denim-id_00000089-04_7_additional.jpg,3,3,0,0,0,0,0,0,3,...,1,MEN,Denim,id_00000089,04_7_additional.jpg,04,7,additional.jpg,Denim,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42539,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,0,0,0,0,0,0,0,1,0,...,1,WOMEN,Tees_Tanks,id_00007979,04_4_full.jpg,04,4,full.jpg,Tees,Tanks
42540,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,0,0,3,0,0,0,0,1,0,...,1,WOMEN,Tees_Tanks,id_00007979,04_7_additional.jpg,04,7,additional.jpg,Tees,Tanks
42541,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,0,4,3,0,0,0,1,1,3,...,1,WOMEN,Tees_Tanks,id_00007981,03_1_front.jpg,03,1,front.jpg,Tees,Tanks
42542,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,0,4,3,0,0,0,1,1,3,...,2,WOMEN,Tees_Tanks,id_00007981,03_3_back.jpg,03,3,back.jpg,Tees,Tanks


In [28]:
def top_modesty(row):
  if row[1] == 3 and row[10] in [2,3,4] and row[12] == 1:
    return 'modest'
  else:
    return 'immodest'



shape_labels['top_modesty'] = shape_labels.apply(top_modesty, axis=1)


In [30]:
women_modest_tops = shape_labels[(shape_labels['top_modesty'] == 'modest') & (shape_labels['Men/Women'] == 'WOMEN')]


In [29]:
women_modest_tops

,img_name,1,2,3,4,5,6,7,8,9,...,Men/Women,Clothing,id,img_num,image_num,img_unit,img_side,clothing,sub_clothing,top_modesty
4962,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_1_front.jpg,02,1,front.jpg,Blouses,Shirts,modest
4963,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_2_side.jpg,02,2,side.jpg,Blouses,Shirts,modest
4965,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,0,0,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_4_full.jpg,02,4,full.jpg,Blouses,Shirts,modest
4966,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,3,4,3,0,0,1,2,1,3,...,WOMEN,Blouses_Shirts,id_00000004,03_1_front.jpg,03,1,front.jpg,Blouses,Shirts,modest
4968,WOMEN-Blouses_Shirts-id_00000004-03_7_addition...,3,4,3,2,4,1,2,1,3,...,WOMEN,Blouses_Shirts,id_00000004,03_7_additional.jpg,03,7,additional.jpg,Blouses,Shirts,modest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42396,WOMEN-Tees_Tanks-id_00007903-11_4_full.jpg,3,3,0,1,0,1,0,1,0,...,WOMEN,Tees_Tanks,id_00007903,11_4_full.jpg,11,4,full.jpg,Tees,Tanks,modest
42397,WOMEN-Tees_Tanks-id_00007903-11_7_additional.jpg,3,4,3,1,0,1,0,1,3,...,WOMEN,Tees_Tanks,id_00007903,11_7_additional.jpg,11,7,additional.jpg,Tees,Tanks,modest
42460,WOMEN-Tees_Tanks-id_00007931-02_7_additional.jpg,3,3,0,0,0,0,0,2,3,...,WOMEN,Tees_Tanks,id_00007931,02_7_additional.jpg,02,7,additional.jpg,Tees,Tanks,modest
42518,WOMEN-Tees_Tanks-id_00007969-03_7_additional.jpg,3,0,1,0,0,1,0,0,3,...,WOMEN,Tees_Tanks,id_00007969,03_7_additional.jpg,03,7,additional.jpg,Tees,Tanks,modest


In [31]:
women_modest_tops = women_modest_tops.rename(columns={0: 'img_name'})

In [32]:
women_modest_tops.to_csv(
    '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_data_csv/women_modest_tops.csv'
)


In [ ]:
women_tops = shape_labels[shape_labels['Men/Women'] == 'WOMEN']


In [ ]:
women_tops

,0,1,2,3,4,5,6,7,8,9,...,Men/Women,Clothing,id,img_num,image_num,img_unit,img_side,clothing,sub_clothing,top_modesty
4962,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_1_front.jpg,02,1,front.jpg,Blouses,Shirts,modest
4963,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_2_side.jpg,02,2,side.jpg,Blouses,Shirts,modest
4964,WOMEN-Blouses_Shirts-id_00000001-02_3_back.jpg,3,0,3,0,0,2,1,0,3,...,WOMEN,Blouses_Shirts,id_00000001,02_3_back.jpg,02,3,back.jpg,Blouses,Shirts,immodest
4965,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,0,0,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_4_full.jpg,02,4,full.jpg,Blouses,Shirts,modest
4966,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,3,4,3,0,0,1,2,1,3,...,WOMEN,Blouses_Shirts,id_00000004,03_1_front.jpg,03,1,front.jpg,Blouses,Shirts,modest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42539,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,0,0,0,0,0,0,0,1,0,...,WOMEN,Tees_Tanks,id_00007979,04_4_full.jpg,04,4,full.jpg,Tees,Tanks,immodest
42540,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,0,0,3,0,0,0,0,1,0,...,WOMEN,Tees_Tanks,id_00007979,04_7_additional.jpg,04,7,additional.jpg,Tees,Tanks,immodest
42541,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,0,4,3,0,0,0,1,1,3,...,WOMEN,Tees_Tanks,id_00007981,03_1_front.jpg,03,1,front.jpg,Tees,Tanks,immodest
42542,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,0,4,3,0,0,0,1,1,3,...,WOMEN,Tees_Tanks,id_00007981,03_3_back.jpg,03,3,back.jpg,Tees,Tanks,immodest


In [ ]:
modest_tops_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_tops/'
women_modest_tops_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_tops/women_modest_tops/'
men_modest_tops_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_tops/men_modest_tops/'
images_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/images/'
women_tops_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_tops/women_tops/'

os.makedirs(modest_tops_dir, exist_ok=True)
os.makedirs(women_modest_tops_dir, exist_ok=True)
os.makedirs(men_modest_tops_dir, exist_ok=True)
os.makedirs(women_tops_dir, exist_ok=True )


KeyboardInterrupt: 

In [ ]:
women_tops.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_tops.csv')

In [ ]:
import shutil

In [ ]:
# for index, row in women_modest_tops.iterrows():
#   image_filename = str(row[0])
#   source_path = os.path.join(images_dir, image_filename)
#   destination_path = os.path.join(women_modest_tops_dir, image_filename)

#   if os.path.exists(source_path):
#     shutil.copy2(source_path, destination_path)
#     print(f"Copied '{image_filename}' to '{women_modest_tops_dir}'")
#   else:
#     print(f"Image file not found: {source_path}")


In [ ]:
shape_labels['Clothing'].unique()

array(['Denim', 'Jackets_Vests', 'Pants', 'Shirts_Polos', 'Shorts',
       'Suiting', 'Sweaters', 'Sweatshirts_Hoodies', 'Tees_Tanks',
       'Blouses_Shirts', 'Cardigans', 'Dresses', 'Graphic_Tees',
       'Jackets_Coats', 'Leggings', 'Rompers_Jumpsuits', 'Skirts'],
      dtype=object)

In [ ]:
shape_labels.loc[3]

,3
0,MEN-Denim-id_00000089-03_7_additional.jpg
1,1
2,3
3,0
4,0
5,0
6,0
7,0
8,0
9,3


In [ ]:
shape_labels['Clothing'].unique()

array(['Denim', 'Jackets_Vests', 'Pants', 'Shirts_Polos', 'Shorts',
       'Suiting', 'Sweaters', 'Sweatshirts_Hoodies', 'Tees_Tanks',
       'Blouses_Shirts', 'Cardigans', 'Dresses', 'Graphic_Tees',
       'Jackets_Coats', 'Leggings', 'Rompers_Jumpsuits', 'Skirts'],
      dtype=object)

In [40]:

def bottom_modesty(row):
    lower_length = row[2]  # 0=three-point, 1=medium-short, 2=three-quarter, 3=long, 4=NA
    leg_coverage = row[3]  # 0=no, 1=socks, 2=leggings, 3=NA

    # Long clothing covers everything — always modest
    if lower_length == 3:
        return 'modest'

    # Shorter clothing is only modest if leggings underneath
    if lower_length in [0, 1, 2] and leg_coverage == 2:
        return 'modest'

    return 'immodest'

shape_labels['bottom_modesty'] = shape_labels.apply(bottom_modesty, axis=1)
women_modest_bottoms = shape_labels[(shape_labels['bottom_modesty'] == 'modest') & (shape_labels['Men/Women'] == 'WOMEN')]


modest_bottoms_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_bottoms/'
women_modest_bottoms_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_bottoms/women_modest_bottoms/'
men_modest_bottoms_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/modest_bottoms/men_modest_bottoms/'
# women_images_dir = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/women_images/'

# os.makedirs(modest_bottoms_dir, exist_ok=True)
# os.makedirs(women_modest_bottoms_dir, exist_ok=True)
# os.makedirs(men_modest_bottoms_dir, exist_ok=True)

# for index, row in women_modest_bottoms.iterrows():
#   image_filename = str(row[0])
#   source_path = os.path.join(images_dir, image_filename)
#   destination_path = os.path.join(women_modest_bottoms_dir, image_filename)

#   if os.path.exists(source_path):
#     shutil.copy2(source_path, destination_path)
#     print(f"Copied '{image_filename}' to '{women_modest_bottoms_dir}'")
#   else:
#     print(f"Image file not found: {source_path}")




In [41]:
women_modest_bottoms

,0,1,2,3,4,5,6,7,8,9,...,Clothing,id,img_num,image_num,img_unit,img_side,clothing,sub_clothing,top_modesty,bottom_modesty
4967,WOMEN-Blouses_Shirts-id_00000004-03_3_back.jpg,3,3,0,0,0,1,2,2,3,...,Blouses_Shirts,id_00000004,03_3_back.jpg,03,3,back.jpg,Blouses,Shirts,immodest,modest
4974,WOMEN-Blouses_Shirts-id_00000005-02_4_full.jpg,3,3,0,1,0,1,1,1,0,...,Blouses_Shirts,id_00000005,02_4_full.jpg,02,4,full.jpg,Blouses,Shirts,immodest,modest
4983,WOMEN-Blouses_Shirts-id_00000024-04_4_full.jpg,2,3,0,0,0,1,0,1,1,...,Blouses_Shirts,id_00000024,04_4_full.jpg,04,4,full.jpg,Blouses,Shirts,immodest,modest
4986,WOMEN-Blouses_Shirts-id_00000025-05_3_back.jpg,0,3,0,0,0,0,1,1,3,...,Blouses_Shirts,id_00000025,05_3_back.jpg,05,3,back.jpg,Blouses,Shirts,immodest,modest
4991,WOMEN-Blouses_Shirts-id_00000028-01_7_addition...,0,3,0,0,2,0,0,1,0,...,Blouses_Shirts,id_00000028,01_7_additional.jpg,01,7,additional.jpg,Blouses,Shirts,immodest,modest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42499,WOMEN-Tees_Tanks-id_00007962-05_7_additional.jpg,1,3,0,1,0,0,1,1,1,...,Tees_Tanks,id_00007962,05_7_additional.jpg,05,7,additional.jpg,Tees,Tanks,immodest,modest
42503,WOMEN-Tees_Tanks-id_00007962-07_7_additional.jpg,1,3,0,1,0,0,1,1,1,...,Tees_Tanks,id_00007962,07_7_additional.jpg,07,7,additional.jpg,Tees,Tanks,immodest,modest
42507,WOMEN-Tees_Tanks-id_00007962-08_7_additional.jpg,1,3,0,1,0,0,1,1,1,...,Tees_Tanks,id_00007962,08_7_additional.jpg,08,7,additional.jpg,Tees,Tanks,immodest,modest
42511,WOMEN-Tees_Tanks-id_00007962-09_7_additional.jpg,1,3,0,1,0,0,1,1,1,...,Tees_Tanks,id_00007962,09_7_additional.jpg,09,7,additional.jpg,Tees,Tanks,immodest,modest


In [33]:
women_modest_tops.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_modest_tops.csv')

In [37]:
print(women_modest_tops['Clothing'].value_counts())


Clothing
Sweaters               1344
Tees_Tanks              927
Blouses_Shirts          848
Jackets_Coats           665
Dresses                 471
Cardigans               440
Sweatshirts_Hoodies     343
Shorts                  108
Skirts                   96
Rompers_Jumpsuits        74
Denim                    44
Graphic_Tees             42
Pants                    39
Leggings                 19
Name: count, dtype: int64


In [42]:
print(women_modest_bottoms['Clothing'].value_counts())


Clothing
Tees_Tanks             1475
Dresses                1254
Blouses_Shirts          961
Sweaters                650
Rompers_Jumpsuits       543
Pants                   485
Jackets_Coats           314
Cardigans               308
Denim                   137
Graphic_Tees            128
Skirts                  112
Sweatshirts_Hoodies     109
Leggings                101
Shorts                   25
Name: count, dtype: int64


In [43]:
# filter out the the tops and bottoms seperately

TOP_CATEGORIES = [
    'Blouses_Shirts', 'Cardigans', 'Dresses', 'Graphic_Tees',
    'Jackets_Coats', 'Rompers_Jumpsuits', 'Sweaters',
    'Sweatshirts_Hoodies', 'Tees_Tanks'
]

BOTTOM_CATEGORIES = [
    'Denim', 'Leggings', 'Pants', 'Shorts', 'Skirts'
]

women_modest_tops = shape_labels[
    (shape_labels['top_modesty'] == 'modest') &
    (shape_labels['Men/Women'] == 'WOMEN') &
    (shape_labels['Clothing'].isin(TOP_CATEGORIES))
]
women_modest_tops = women_modest_tops.rename(columns={0: 'img_name'})
women_modest_tops.to_csv(
    '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_modest_tops.csv'
)
print(f"Modest tops: {len(women_modest_tops)}")
print(women_modest_tops['Clothing'].value_counts())

women_modest_bottoms = shape_labels[
    (shape_labels['bottom_modesty'] == 'modest') &
    (shape_labels['Men/Women'] == 'WOMEN') &
    (shape_labels['Clothing'].isin(BOTTOM_CATEGORIES))
]
women_modest_bottoms = women_modest_bottoms.rename(columns={0: 'img_name'})
women_modest_bottoms.to_csv(
    '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_modest_bottoms.csv'
)
print(f"Modest bottoms: {len(women_modest_bottoms)}")
print(women_modest_bottoms['Clothing'].value_counts())


Modest tops: 5154
Clothing
Sweaters               1344
Tees_Tanks              927
Blouses_Shirts          848
Jackets_Coats           665
Dresses                 471
Cardigans               440
Sweatshirts_Hoodies     343
Rompers_Jumpsuits        74
Graphic_Tees             42
Name: count, dtype: int64
Modest bottoms: 860
Clothing
Pants       485
Denim       137
Skirts      112
Leggings    101
Shorts       25
Name: count, dtype: int64


In [ ]:
# three training routes i will take:
# 1. train all the data on both the top data and the bottom data
# 2. train the top data only
# 3. train the bottom data only to be identified as modest/immodest
# this is a multi-label multimodal classification problem.

shap_annons = {
    "sleeve_length": {0: "sleeveless", 1: "short-sleeve", 2: "medium-sleeve", 3: "long-sleeve", 4: "not long-sleeve", 5: "NA"},
    "lower_clothing_length": {0: "three-point", 1: "medium short", 2 :"three-quarter", 3: "long", 4: "NA"},
    "socks": {0: "no", 1: "socks", 2: "leggings", 3: "NA"},
    "hat": {0: "no", 1: "yes", 2: "NA"},
    "glasses": {0: "no", 1: "eyeglasses", 2: "sunglasses", 3: "have a glasses in hand or clothes", 4: "NA"},
    "neckwear": {0: "no", 1: "yes", 2: "NA"},
    "wrist_wearing": {0: "no", 1: "yes", 2: "NA"},
    "ring": {0: "no", 1: "yes", 2: "NA"},
    "waist_accessories": {0: "no", 1: "belt", 2: "have a clothing", 3: "hidden", 4: "NA"},
    "neckline": {0: "V-shape", 1: "square", 2: "round", 3: "standing", 4: "lapel", 5: "suspenders", 6: "NA"},
    "outer_clothing_cardigan": {0: "yes", 1: "no", 2: "NA"},
    "upper_clothing_covering_navel": {0: "no", 1: "yes", 2: "NA"}
}

top_modest = {
              'sleeve_length': {3:'long_sleeve'},
              'neckline': {2:'round', 3:'standing', 4:'lapel'},
              'outer_clothing_cardigan': {0:'yes'},
              'upper_clothing_covering_navel': {1:'yes'},
              }

# column number in shape_labels
top_modest = [3, [2,3,4], 0, 1]


bottom_modest = {
    'lower_clothing_length': {3:'long'},
    'socks': {1:'socks'}
    }

bottom_modest = [3, [1, 2]]


In [34]:
women_modest_bottoms['Clothing'].unique()

NameError: name 'women_modest_bottoms' is not defined

In [ ]:
women_modest_bottoms.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_modest_bottoms.csv')

In [ ]:
os.path.exists('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg')

False

## EDA: Fabric

In [ ]:
fabric_labels = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/labels/labels/texture/fabric_ann.txt'
fabric_annons = ['denim','cotton','leather','furry', 'knitted', 'chiffon', 'other', 'NA']


In [ ]:
fabric_labels = pd.read_csv(fabric_labels, sep=' ', header=None)


In [ ]:
fabric_labels.columns = ['img', 'upper_fabric', 'lower_fabric', 'outer_fabric']
fabric_labels


,img,upper_fabric,lower_fabric,outer_fabric
0,MEN-Denim-id_00000080-01_7_additional.jpg,1,1,7
1,MEN-Denim-id_00000089-01_7_additional.jpg,1,1,7
2,MEN-Denim-id_00000089-02_7_additional.jpg,1,1,7
3,MEN-Denim-id_00000089-03_7_additional.jpg,1,1,7
4,MEN-Denim-id_00000089-04_7_additional.jpg,0,1,7
...,...,...,...,...
44091,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,1,1,7
44092,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,1,0,7
44093,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,1,0,7
44094,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,1,0,7


In [ ]:
fabric_labels[['Men/Women', 'Clothing', 'id', 'img_num']] = fabric_labels['img'].str.split('-', expand=True)
# fabric_labels[fabric_labels['Men/Women'] == 'WOMEN']
fabric_labels
women_fabric = fabric_labels[fabric_labels['Men/Women'] == 'WOMEN']


In [ ]:
women_fabric.drop(columns=['img_num', 'Clothing', 'id', 'Men/Women'])

,img,upper_fabric,lower_fabric,outer_fabric
5437,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,5,5,7
5438,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,5,5,7
5439,WOMEN-Blouses_Shirts-id_00000001-02_3_back.jpg,5,5,7
5440,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,5,5,7
5441,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,5,0,7
...,...,...,...,...
44091,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,1,1,7
44092,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,1,0,7
44093,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,1,0,7
44094,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,1,0,7


In [ ]:
women_fabric.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_fabric.csv')

## EDA: Patterns


In [ ]:
pattern_labels = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/labels/labels/texture/pattern_ann.txt'


In [ ]:
pattern_labels = pd.read_csv(pattern_labels, sep=' ', header=None)

In [ ]:
pattern_labels.columns = ['img','upper_color', 'lower_color', 'outer_color']

In [ ]:
pattern_labels[['Men/Women', 'Clothing', 'id', 'img_num']] = pattern_labels['img'].str.split('-', expand=True)
pattern_labels[pattern_labels['Men/Women'] == 'WOMEN']


,img,upper_color,lower_color,outer_color,Men/Women,Clothing,id,img_num
5437,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,3,3,7,WOMEN,Blouses_Shirts,id_00000001,02_1_front.jpg
5438,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,3,3,7,WOMEN,Blouses_Shirts,id_00000001,02_2_side.jpg
5439,WOMEN-Blouses_Shirts-id_00000001-02_3_back.jpg,3,3,7,WOMEN,Blouses_Shirts,id_00000001,02_3_back.jpg
5440,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,3,7,WOMEN,Blouses_Shirts,id_00000001,02_4_full.jpg
5441,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,1,3,7,WOMEN,Blouses_Shirts,id_00000004,03_1_front.jpg
...,...,...,...,...,...,...,...,...
44091,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,3,3,7,WOMEN,Tees_Tanks,id_00007979,04_4_full.jpg
44092,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,3,3,7,WOMEN,Tees_Tanks,id_00007979,04_7_additional.jpg
44093,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,5,3,7,WOMEN,Tees_Tanks,id_00007981,03_1_front.jpg
44094,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,5,3,7,WOMEN,Tees_Tanks,id_00007981,03_3_back.jpg


In [ ]:
pattern_labels.drop(columns=['img_num', 'id', 'Clothing'], inplace=True)

In [ ]:
women_patterns = pattern_labels[pattern_labels['Men/Women'] == 'WOMEN']

In [ ]:
women_patterns

,img,upper_color,lower_color,outer_color,Men/Women
5437,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,3,3,7,WOMEN
5438,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,3,3,7,WOMEN
5439,WOMEN-Blouses_Shirts-id_00000001-02_3_back.jpg,3,3,7,WOMEN
5440,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,3,7,WOMEN
5441,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,1,3,7,WOMEN
...,...,...,...,...,...
44091,WOMEN-Tees_Tanks-id_00007979-04_4_full.jpg,3,3,7,WOMEN
44092,WOMEN-Tees_Tanks-id_00007979-04_7_additional.jpg,3,3,7,WOMEN
44093,WOMEN-Tees_Tanks-id_00007981-03_1_front.jpg,5,3,7,WOMEN
44094,WOMEN-Tees_Tanks-id_00007981-03_3_back.jpg,5,3,7,WOMEN


In [ ]:
women_patterns.to_csv('/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/dataframes/women_patterns.csv')

## EDA: *Landmarks*


In [ ]:
keypoints_loc = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/keypoints/keypoints_loc.txt'

In [ ]:
keypoints_vis = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/keypoints/keypoints_vis.txt'

In [ ]:
column_names = ['img_name',
                        "x_1", "y_1", "x_2", "y_2", "x_3", "y_3", "x_4", "y_4", "x_5", "y_5",
    "x_6", "y_6", "x_7", "y_7", "x_8", "y_8", "x_9", "y_9", "x_10", "y_10",
    "x_11", "y_11", "x_12", "y_12", "x_13", "y_13", "x_14", "y_14", "x_15", "y_15",
    "x_16", "y_16", "x_17", "y_17", "x_18", "y_18", "x_19", "y_19", "x_20", "y_20",
    "x_21", "y_21"]

In [ ]:
keypoints_df = pd.read_csv(keypoints_loc, sep='\s+', header=None, names=column_names)

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_5646/1372871600.py:1: SyntaxWarning: invalid escape sequence '\s'
  keypoints_df = pd.read_csv(keypoints_loc, sep='\s+', header=None, names=column_names)


In [ ]:
keypoints_df

,img_name,x_1,y_1,x_2,y_2,x_3,y_3,x_4,y_4,x_5,...,x_17,y_17,x_18,y_18,x_19,y_19,x_20,y_20,x_21,y_21
0,MEN-Tees_Tanks-id_00007466-05_7_additional.jpg,429,55,382,130,408,106,446,106,469,...,508,524,311,769,449,763,352,1004,402,1028
1,WOMEN-Tees_Tanks-id_00004475-03_7_additional.jpg,417,20,364,93,387,79,424,89,448,...,538,522,352,761,476,767,382,986,472,989
2,WOMEN-Dresses-id_00001697-01_4_full.jpg,297,20,248,94,274,87,317,94,335,...,377,546,246,749,440,746,226,1007,397,997
3,WOMEN-Shorts-id_00006051-04_4_full.jpg,387,32,332,101,363,91,399,96,420,...,480,531,336,717,385,735,352,862,337,975
4,WOMEN-Dresses-id_00002993-03_4_full.jpg,394,27,343,101,372,101,411,103,434,...,455,530,351,751,405,749,371,882,400,1022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12697,WOMEN-Tees_Tanks-id_00000276-06_4_full.jpg,340,23,299,109,324,99,366,99,390,...,490,499,331,755,469,763,328,989,369,975
12698,WOMEN-Dresses-id_00005006-03_4_full.jpg,-1,-1,346,75,364,60,403,61,437,...,492,518,334,732,365,763,476,987,584,885
12699,WOMEN-Tees_Tanks-id_00005461-01_1_front.jpg,267,32,250,116,259,103,291,92,335,...,423,503,282,716,399,721,335,968,234,905
12700,WOMEN-Dresses-id_00001761-01_4_full.jpg,435,39,398,124,418,117,459,113,481,...,535,552,371,786,448,744,483,863,394,977


In [ ]:
women_modest_tops

,img_name,1,2,3,4,5,6,7,8,9,...,Men/Women,Clothing,id,img_num,image_num,img_unit,img_side,clothing,sub_clothing,top_modesty
4962,WOMEN-Blouses_Shirts-id_00000001-02_1_front.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_1_front.jpg,02,1,front.jpg,Blouses,Shirts,modest
4963,WOMEN-Blouses_Shirts-id_00000001-02_2_side.jpg,3,0,3,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_2_side.jpg,02,2,side.jpg,Blouses,Shirts,modest
4965,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,0,0,0,0,0,1,1,3,...,WOMEN,Blouses_Shirts,id_00000001,02_4_full.jpg,02,4,full.jpg,Blouses,Shirts,modest
4966,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,3,4,3,0,0,1,2,1,3,...,WOMEN,Blouses_Shirts,id_00000004,03_1_front.jpg,03,1,front.jpg,Blouses,Shirts,modest
4968,WOMEN-Blouses_Shirts-id_00000004-03_7_addition...,3,4,3,2,4,1,2,1,3,...,WOMEN,Blouses_Shirts,id_00000004,03_7_additional.jpg,03,7,additional.jpg,Blouses,Shirts,modest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42396,WOMEN-Tees_Tanks-id_00007903-11_4_full.jpg,3,3,0,1,0,1,0,1,0,...,WOMEN,Tees_Tanks,id_00007903,11_4_full.jpg,11,4,full.jpg,Tees,Tanks,modest
42397,WOMEN-Tees_Tanks-id_00007903-11_7_additional.jpg,3,4,3,1,0,1,0,1,3,...,WOMEN,Tees_Tanks,id_00007903,11_7_additional.jpg,11,7,additional.jpg,Tees,Tanks,modest
42460,WOMEN-Tees_Tanks-id_00007931-02_7_additional.jpg,3,3,0,0,0,0,0,2,3,...,WOMEN,Tees_Tanks,id_00007931,02_7_additional.jpg,02,7,additional.jpg,Tees,Tanks,modest
42518,WOMEN-Tees_Tanks-id_00007969-03_7_additional.jpg,3,0,1,0,0,1,0,0,3,...,WOMEN,Tees_Tanks,id_00007969,03_7_additional.jpg,03,7,additional.jpg,Tees,Tanks,modest


In [ ]:
modest_women_keypoints = pd.merge(women_modest_tops, keypoints_df, on='img_name', how='inner')

In [ ]:
modest_women_keypoints

,img_name,1,2,3,4,5,6,7,8,9,...,x_17,y_17,x_18,y_18,x_19,y_19,x_20,y_20,x_21,y_21
0,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,0,0,0,0,0,1,1,3,...,471,500,246,773,384,767,448,935,405,992
1,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,3,4,3,0,0,1,2,1,3,...,631,616,421,923,541,919,-1,-1,-1,-1
2,WOMEN-Blouses_Shirts-id_00000230-01_4_full.jpg,3,0,2,1,0,0,0,1,3,...,475,189,348,751,438,770,311,953,381,1000
3,WOMEN-Blouses_Shirts-id_00000350-01_4_full.jpg,3,3,0,1,0,1,1,1,0,...,443,539,286,757,329,745,356,965,279,1013
4,WOMEN-Blouses_Shirts-id_00000350-02_4_full.jpg,3,3,0,0,0,1,1,1,3,...,497,504,370,742,462,732,395,985,384,986
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2159,WOMEN-Tees_Tanks-id_00007903-10_7_additional.jpg,3,3,0,1,0,1,0,1,0,...,439,529,318,742,338,746,364,915,289,1031
2160,WOMEN-Tees_Tanks-id_00007903-11_4_full.jpg,3,3,0,1,0,1,0,1,0,...,480,167,343,768,479,791,395,1015,286,954
2161,WOMEN-Tees_Tanks-id_00007931-02_7_additional.jpg,3,3,0,0,0,0,0,2,3,...,454,509,247,790,364,787,478,867,368,1028
2162,WOMEN-Tees_Tanks-id_00007969-03_7_additional.jpg,3,0,1,0,0,1,0,0,3,...,470,526,343,718,470,744,387,945,331,861


## EDA Segmentation

In [ ]:
from pathlib import Path

In [ ]:
segments = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/segm/segm'


In [ ]:
_women_modest_segm = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/segm/_women_modest_segm/'
# os.makedirs(_women_modest_segm, exist_ok=True)


In [ ]:
modest_women_keypoints

,img_name,1,2,3,4,5,6,7,8,9,...,x_17,y_17,x_18,y_18,x_19,y_19,x_20,y_20,x_21,y_21
0,WOMEN-Blouses_Shirts-id_00000001-02_4_full.jpg,3,0,0,0,0,0,1,1,3,...,471,500,246,773,384,767,448,935,405,992
1,WOMEN-Blouses_Shirts-id_00000004-03_1_front.jpg,3,4,3,0,0,1,2,1,3,...,631,616,421,923,541,919,-1,-1,-1,-1
2,WOMEN-Blouses_Shirts-id_00000230-01_4_full.jpg,3,0,2,1,0,0,0,1,3,...,475,189,348,751,438,770,311,953,381,1000
3,WOMEN-Blouses_Shirts-id_00000350-01_4_full.jpg,3,3,0,1,0,1,1,1,0,...,443,539,286,757,329,745,356,965,279,1013
4,WOMEN-Blouses_Shirts-id_00000350-02_4_full.jpg,3,3,0,0,0,1,1,1,3,...,497,504,370,742,462,732,395,985,384,986
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2159,WOMEN-Tees_Tanks-id_00007903-10_7_additional.jpg,3,3,0,1,0,1,0,1,0,...,439,529,318,742,338,746,364,915,289,1031
2160,WOMEN-Tees_Tanks-id_00007903-11_4_full.jpg,3,3,0,1,0,1,0,1,0,...,480,167,343,768,479,791,395,1015,286,954
2161,WOMEN-Tees_Tanks-id_00007931-02_7_additional.jpg,3,3,0,0,0,0,0,2,3,...,454,509,247,790,364,787,478,867,368,1028
2162,WOMEN-Tees_Tanks-id_00007969-03_7_additional.jpg,3,0,1,0,0,1,0,0,3,...,470,526,343,718,470,744,387,945,331,861


In [ ]:
import matplotlib.pyplot as plt
import os

# Get the first image name from the modest_women_keypoints DataFrame
first_image_name = modest_women_keypoints['img_name'].iloc[0]

# Construct the full image path
image_path = os.path.join(images_dir, first_image_name)

# Check if the image exists and display it
if os.path.exists(image_path):
    img = plt.imread(image_path)
    plt.imshow(img)
    plt.axis('off') # Hide axes
    plt.title(first_image_name)
    plt.show()
else:
    print(f"Image not found at {image_path}")

In [ ]:
women_modest_tops['Clothing'].unique()

In [ ]:

unique_clothing_items = women_modest_tops['Clothing'].unique().tolist()
unique_clothing_items


In [ ]:
women_modest_tops['Clothing'].nunique()

In [ ]:
women_modest_tops

In [ ]:
sdf

In [ ]:
hjb

## Women's Images

### Extracting Women's Images

In [ ]:
women_images_folder = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/women_images'

In [ ]:
images_folder_path = '/content/drive/MyDrive/PersonalProjects/modesty/Deepfashion/images/images'

In [ ]:
import shutil

for image in os.listdir(images_folder_path):
  if image.startswith("WOMEN"):
    source_path = os.path.join(images_folder_path, image)
    destination_path = os.path.join(women_images_folder, image)
    shutil.copy2(source_path, destination_path)

In [ ]:
# for image_filename in os.listdir(women_images_folder):
#   parts = image_filename.split('-')
#   if len(parts) > 1:
#     phrases = []
#     for i in range(1, len(parts)):
#       if '-' in parts[i]:
#         phrases.extend(parts[i].split('-'))
#     print(image_filename, ":", phrases)

In [ ]:
# fabric_labels[['Men/Women', 'Clothing', 'id', 'img_num']] = fabric_labels[0].str.split('-', expand=True)
# fabric_labels[fabric_labels['Men/Women'] == 'WOMEN']

In [ ]:
# pattern_labels[['Men/Women', 'Clothing', 'id', 'img_num']] = pattern_labels[0].str.split('-', expand=True)
# pattern_labels[pattern_labels['Men/Women'] == 'WOMEN']

In [ ]:
# keep editing the data and understanding, see whether you want to add shape annotations or not into the shape_labels dataframe

**EDA**: Women's Images

In [ ]:
women_shape_labels = shape_labels[shape_labels['Men/Women'] == 'WOMEN']

In [ ]:
women_shape_labels.columns

In [ ]:
shape_labels